# Phase 0 - Tokenizer Discovery

This notebook is the single source for Phase 0 tokenizer discovery.

Goals:
- Load the `ytu-ce-cosmos/turkish-gpt2` tokenizer.
- Classify how Turkish verb forms are split into tokens.
- Build the pilot overt-subject vs. pro-drop position reference.
- Save reproducible local CSV outputs for later phases.

Output policy: files under `results/` are generated artifacts.

## 1. Setup

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.dataset import VERB_FORMS, Person, Tense, Verb, get_pilot
import core.utils as core_utils
core_utils = importlib.reload(core_utils)
classify_verb_tokenization = core_utils.classify_verb_tokenization
compare_tokenizations = core_utils.compare_tokenizations
from core import viz

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

RESULTS_DIR = PROJECT_ROOT / "results"
SAVE_RESULTS = True

print(f"Project root: {PROJECT_ROOT}")

d:\Projects\ProDropLens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: d:\Projects\ProDropLens


## 2. Load Tokenizer

In [2]:
MODEL_NAME = "ytu-ce-cosmos/turkish-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length}")

Tokenizer: ytu-ce-cosmos/turkish-gpt2
Vocabulary size: 50,257
Model max length: 1000000000000000019884624838656


## 3. Verb Form Tokenization

Each verb form is classified into one of three analysis-relevant tokenization types:

- `A_merged`: person information is merged into a verb token.
- `B_suffix_split`: person suffix is a separate token.
- `C_multi_split`: multi-token form without a clean suffix split.

In [3]:
classification_rows = []

for verb in Verb:
    for tense in Tense:
        for person in Person:
            form = VERB_FORMS[verb][tense][person]
            info = classify_verb_tokenization(tokenizer, form)
            classification_rows.append({
                "verb": verb.value,
                "tense": tense.value,
                "person": person.value,
                "form": form,
                "type": info["type"],
                "n_tokens": info["n_tokens"],
                "tokens": " | ".join(info["tokens"]),
                "suffix_token": info["suffix_token"] or "-",
            })

verb_classification_df = pd.DataFrame(classification_rows)

display(verb_classification_df)

type_summary_df = (
    verb_classification_df["type"]
    .value_counts()
    .rename_axis("type")
    .reset_index(name="count")
)
display(type_summary_df)

,verb,tense,person,form,type,n_tokens,tokens,suffix_token
0,gitmek,simdiki,1s,gidiyorum,A_merged,1,gidiyorum,-
1,gitmek,simdiki,2s,gidiyorsun,B_suffix_split,2,gidiyor | sun,sun
2,gitmek,simdiki,3s,gidiyor,A_merged,1,gidiyor,-
3,gitmek,simdiki,1p,gidiyoruz,A_merged,1,gidiyoruz,-
4,gitmek,simdiki,2p,gidiyorsunuz,B_suffix_split,2,gidiyor | sunuz,sunuz
...,...,...,...,...,...,...,...,...
67,yapmak,genis,2s,yaparsın,A_merged,1,yaparsın,-
68,yapmak,genis,3s,yapar,A_merged,1,yapar,-
69,yapmak,genis,1p,yaparız,A_merged,1,yaparız,-
70,yapmak,genis,2p,yaparsınız,A_merged,1,yaparsınız,-


,type,count
0,A_merged,52
1,B_suffix_split,15
2,C_multi_split,5


In [4]:
type_b_df = verb_classification_df[verb_classification_df["type"] == "B_suffix_split"]
type_c_df = verb_classification_df[verb_classification_df["type"] == "C_multi_split"]

print("TYPE B forms: suffix is a separate token")
display(type_b_df[["verb", "tense", "person", "form", "tokens", "suffix_token"]])

print("TYPE C forms: inspect case by case")
display(type_c_df[["verb", "tense", "person", "form", "tokens"]])

TYPE B forms: suffix is a separate token


,verb,tense,person,form,tokens,suffix_token
1,gitmek,simdiki,2s,gidiyorsun,gidiyor | sun,sun
4,gitmek,simdiki,2p,gidiyorsunuz,gidiyor | sunuz,sunuz
13,gitmek,gelecek,2s,gideceksin,gidecek | sin,sin
16,gitmek,gelecek,2p,gideceksiniz,gidecek | siniz,siniz
17,gitmek,gelecek,3p,gidecekler,gidecek | ler,ler
21,gitmek,genis,1p,gideriz,gider | iz,iz
25,gelmek,simdiki,2s,geliyorsun,geliyor | sun,sun
28,gelmek,simdiki,2p,geliyorsunuz,geliyor | sunuz,sunuz
37,gelmek,gelecek,2s,geleceksin,gelecek | sin,sin
40,gelmek,gelecek,2p,geleceksiniz,gelecek | siniz,siniz


TYPE C forms: inspect case by case


,verb,tense,person,form,tokens
7,gitmek,gecmis,2s,gittin,git | tin
10,gitmek,gecmis,2p,gittiniz,git | tiniz
19,gitmek,genis,2s,gidersin,gid | ersin
22,gitmek,genis,2p,gidersiniz,gid | ersiniz
36,gelmek,gelecek,1s,geleceğim,gel | eceğim


## 4. Pilot Minimal-Pair Position Reference

For each pilot pair, compare the overt-subject sentence with the pro-drop sentence and record the verb span positions.

In [5]:
pilot_pairs = get_pilot()
position_rows = []
comparison_examples = []


def fmt_list(values):
    if values is None:
        return "-"
    values = list(values)
    if not values:
        return "-"
    return " | ".join(str(v) for v in values)


def form_encoding(form):
    info = classify_verb_tokenization(tokenizer, form)
    return {
        "type": info["type"],
        "n_tokens": info["n_tokens"],
        "token_ids": info["token_ids"],
        "tokens": info["tokens"],
        "suffix_token": info["suffix_token"] or "-",
    }


def token_at_span_pos(sentence_info, pos):
    span = sentence_info["verb_span"]
    if span is None or pos is None:
        return None, None
    offset = pos - span.start
    if offset < 0 or offset >= span.n_tokens:
        return None, None
    return span.token_ids[offset], span.token_strs[offset].strip()


def choose_compare_mode(target_form, contrast_form, target_info, contrast_info):
    if target_form == contrast_form:
        return "self_contrast_baseline"
    if target_info["n_tokens"] == 1 and contrast_info["n_tokens"] == 1:
        return "single_token_logit_diff"
    if target_info["type"] == "B_suffix_split":
        return "suffix_probe_plus_sequence_logprob"
    return "sequence_logprob_case_by_case"


for pair in pilot_pairs:
    comparison = compare_tokenizations(
        tokenizer,
        overt_text=pair.overt_text,
        prodrop_text=pair.prodrop_text,
        verb_form=pair.target_form,
        subject=pair.subject_token,
    )
    overt = comparison["overt"]
    prodrop = comparison["prodrop"]
    target_info = form_encoding(pair.target_form)
    contrast_info = form_encoding(pair.contrast_form)

    overt_primary_id, overt_primary_tok = token_at_span_pos(overt, overt["primary_probe_pos"])
    prodrop_primary_id, prodrop_primary_tok = token_at_span_pos(prodrop, prodrop["primary_probe_pos"])
    same_prefix_as_contrast = (
        len(target_info["token_ids"]) >= len(contrast_info["token_ids"])
        and target_info["token_ids"][:len(contrast_info["token_ids"])] == contrast_info["token_ids"]
    )

    position_rows.append({
        "id": pair.id,
        "person": pair.person.value,
        "subject": pair.subject_token,
        "target_form": pair.target_form,
        "contrast_form": pair.contrast_form,
        "compare_mode": choose_compare_mode(
            pair.target_form, pair.contrast_form, target_info, contrast_info
        ),
        "target_type": target_info["type"],
        "target_n_tokens": target_info["n_tokens"],
        "target_tokens": fmt_list(target_info["tokens"]),
        "target_token_ids": fmt_list(target_info["token_ids"]),
        "target_suffix_token": target_info["suffix_token"],
        "contrast_type": contrast_info["type"],
        "contrast_n_tokens": contrast_info["n_tokens"],
        "contrast_tokens": fmt_list(contrast_info["tokens"]),
        "contrast_token_ids": fmt_list(contrast_info["token_ids"]),
        "is_self_contrast": pair.target_form == pair.contrast_form,
        "same_prefix_as_contrast": same_prefix_as_contrast,
        "overt_text": pair.overt_text,
        "prodrop_text": pair.prodrop_text,
        "overt_tokens": fmt_list(overt["token_strs"]),
        "overt_token_ids": fmt_list(overt["token_ids"]),
        "prodrop_tokens": fmt_list(prodrop["token_strs"]),
        "prodrop_token_ids": fmt_list(prodrop["token_ids"]),
        "overt_n_tokens": overt["n_tokens"],
        "prodrop_n_tokens": prodrop["n_tokens"],
        "overt_subject_pos": overt["subject_pos"],
        "prodrop_subject_pos": prodrop["subject_pos"],
        "overt_verb_start": overt["verb_pos"],
        "overt_verb_end": overt["verb_end"],
        "prodrop_verb_start": prodrop["verb_pos"],
        "prodrop_verb_end": prodrop["verb_end"],
        "verb_offset": comparison["verb_offset"],
        "suffix_split": overt["suffix_split"],
        "suffix_pos_overt": overt["suffix_pos"],
        "suffix_pos_prodrop": prodrop["suffix_pos"],
        "target_primary_token_id": overt_primary_id,
        "target_primary_token": overt_primary_tok,
        "overt_primary_probe_pos": overt["primary_probe_pos"],
        "overt_primary_read_pos": overt["primary_read_pos"],
        "prodrop_primary_probe_pos": prodrop["primary_probe_pos"],
        "prodrop_primary_read_pos": prodrop["primary_read_pos"],
        "overt_first_token_read_pos": overt["first_token_read_pos"],
        "prodrop_first_token_read_pos": prodrop["first_token_read_pos"],
        "overt_verb_token_read_positions": fmt_list(overt["verb_token_read_positions"]),
        "prodrop_verb_token_read_positions": fmt_list(prodrop["verb_token_read_positions"]),
        "overt_subject_to_verb_start": (
            overt["verb_pos"] - overt["subject_pos"]
            if overt["verb_pos"] is not None and overt["subject_pos"] is not None
            else None
        ),
        "overt_subject_to_primary_read": (
            overt["primary_read_pos"] - overt["subject_pos"]
            if overt["primary_read_pos"] is not None and overt["subject_pos"] is not None
            else None
        ),
    })

    comparison_examples.append((pair, overt, prodrop))

position_reference_df = pd.DataFrame(position_rows)
display(position_reference_df)


,id,person,subject,target_form,contrast_form,compare_mode,target_type,target_n_tokens,target_tokens,target_token_ids,...,overt_primary_probe_pos,overt_primary_read_pos,prodrop_primary_probe_pos,prodrop_primary_read_pos,overt_first_token_read_pos,prodrop_first_token_read_pos,overt_verb_token_read_positions,prodrop_verb_token_read_positions,overt_subject_to_verb_start,overt_subject_to_primary_read
0,pilot_gitmek_simdiki_1s,1s,Ben,gidiyorum,gidiyor,single_token_logit_diff,A_merged,1,gidiyorum,23301,...,4,3,3,2,3,2,3,2,4,3
1,pilot_gitmek_simdiki_2s,2s,Sen,gidiyorsun,gidiyor,suffix_probe_plus_sequence_logprob,B_suffix_split,2,gidiyor | sun,6215 | 833,...,5,4,4,3,3,2,3 | 4,2 | 3,4,4
2,pilot_gitmek_simdiki_3s,3s,O,gidiyor,gidiyor,self_contrast_baseline,A_merged,1,gidiyor,6215,...,4,3,3,2,3,2,3,2,4,3
3,pilot_gitmek_simdiki_1p,1p,Biz,gidiyoruz,gidiyor,single_token_logit_diff,A_merged,1,gidiyoruz,19015,...,4,3,3,2,3,2,3,2,4,3
4,pilot_gitmek_simdiki_2p,2p,Siz,gidiyorsunuz,gidiyor,suffix_probe_plus_sequence_logprob,B_suffix_split,2,gidiyor | sunuz,6215 | 1711,...,5,4,4,3,3,2,3 | 4,2 | 3,4,4
5,pilot_gitmek_simdiki_3p,3p,Onlar,gidiyorlar,gidiyor,single_token_logit_diff,A_merged,1,gidiyorlar,41311,...,4,3,3,2,3,2,3,2,4,3


## 5. Visual Checks

In [6]:
example_pair, example_overt, example_prodrop = comparison_examples[0]

fig = viz.plot_tokenization_comparison(example_overt, example_prodrop, example_pair.id)
fig.show()

In [7]:
offsets = dict(zip(position_reference_df["id"], position_reference_df["verb_offset"]))
fig = viz.plot_position_offset_summary(offsets)
fig.show()

## 6. Analysis Implications

In [8]:
n_total = len(verb_classification_df)
n_type_a = int((verb_classification_df["type"] == "A_merged").sum())
n_type_b = int((verb_classification_df["type"] == "B_suffix_split").sum())
n_type_c = int((verb_classification_df["type"] == "C_multi_split").sum())

analysis_strategy = pd.DataFrame([
    {
        "case": "A_merged",
        "count": n_type_a,
        "share": f"{n_type_a / n_total:.1%}",
        "phase_1_2_position": "Use the token before VerbSpan.start to read logits for the merged verb token.",
        "attention_position": "Inspect subject -> verb context and attention from VerbSpan.start/end to subject_pos.",
    },
    {
        "case": "B_suffix_split",
        "count": n_type_b,
        "share": f"{n_type_b / n_total:.1%}",
        "phase_1_2_position": "Use VerbSpan.primary_read_pos to read the suffix token; also score the full token sequence.",
        "attention_position": "Inspect suffix prediction, stem-to-subject attention, and subject influence before the suffix.",
    },
    {
        "case": "C_multi_split",
        "count": n_type_c,
        "share": f"{n_type_c / n_total:.1%}",
        "phase_1_2_position": "Score the full VerbSpan token sequence and inspect the final token case by case.",
        "attention_position": "Inspect all verb-span tokens; do not assume a clean suffix boundary.",
    },
])

display(analysis_strategy)


,case,count,share,phase_1_2_position,attention_position
0,A_merged,52,72.2%,Use the token before VerbSpan.start to read logits for the merged verb token.,Inspect subject -> verb context and attention from VerbSpan.start/end to subject_pos.
1,B_suffix_split,15,20.8%,Use VerbSpan.primary_read_pos to read the suffix token; also score the full token sequence.,"Inspect suffix prediction, stem-to-subject attention, and subject influence before the suffix."
2,C_multi_split,5,6.9%,Score the full VerbSpan token sequence and inspect the final token case by case.,Inspect all verb-span tokens; do not assume a clean suffix boundary.


## 7. Save Local Outputs

The CSV files are reproducible outputs for later phases. They are written to `results/`, which is ignored by Git.

In [9]:
if SAVE_RESULTS:
    RESULTS_DIR.mkdir(exist_ok=True)

    verb_classification_path = RESULTS_DIR / "phase0_verb_classification.csv"
    position_reference_path = RESULTS_DIR / "phase0_position_reference.csv"

    verb_classification_df.to_csv(verb_classification_path, index=False)
    position_reference_df.to_csv(position_reference_path, index=False)

    print(f"Saved: {verb_classification_path}")
    print(f"Saved: {position_reference_path}")
else:
    print("SAVE_RESULTS is False; no files written.")

Saved: d:\Projects\ProDropLens\results\phase0_verb_classification.csv
Saved: d:\Projects\ProDropLens\results\phase0_position_reference.csv
